In [1]:
# kafka_case_test

In [3]:
import requests
import json
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 1. Список всех бирж
url_list = "https://api.coingecko.com/api/v3/exchanges/list"
exchanges = requests.get(url_list, verify=False).json()

print(f"Всего бирж: {len(exchanges)}")
print(exchanges[:3])

# 2. Объём торгов за 1 день для binance
exchange_id = "binance"
url_volume = f"https://api.coingecko.com/api/v3/exchanges/{exchange_id}/volume_chart"
params = {"days": "1"}

volume_data = requests.get(url_volume, params=params, verify=False).json()

print(f"\nКоличество точек за 1 день: {len(volume_data)}")
print(volume_data[:3])

Всего бирж: 1504
[{'id': '10kswap-starknet-alpha', 'name': '10KSwap'}, {'id': '9inch', 'name': '9inch'}, {'id': '9mm-v3', 'name': '9mm V3 (Pulsechain)'}]

Количество точек за 1 день: 144
[[1785758400000, '65017.6646891814869453'], [1785759000000, '65017.6646891814869453'], [1785759600000, '65617.6588172166528521']]


In [4]:
import requests
import json
import time
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def get_exchanges_list():
    url = "https://api.coingecko.com/api/v3/exchanges/list"
    return requests.get(url, verify=False).json()

def volume_generator(exchanges, max_exchanges=None):
    """
    Генератор, который для каждой биржи:
    - запрашивает объём торгов за 1 день
    - формирует json {exchange_name: volume_data}
    - yield'ит закодированные bytes
    """
    count = 0
    for exchange in exchanges:
        if max_exchanges and count >= max_exchanges:
            break
            
        exchange_id = exchange['id']
        exchange_name = exchange['name']
        
        try:
            url = f"https://api.coingecko.com/api/v3/exchanges/{exchange_id}/volume_chart"
            params = {"days": "1"}
            volume_data = requests.get(url, params=params, verify=False).json()
            
            # Формируем нужную структуру
            payload = {exchange_name: volume_data}
            
            # Кодируем в bytes
            message_bytes = json.dumps(payload).encode('utf-8')
            
            yield message_bytes
            
            count += 1
            time.sleep(1.2)   # чтобы не упереться в rate limit CoinGecko
            
        except Exception as e:
            print(f"Ошибка на бирже {exchange_name}: {e}")
            continue

In [5]:
exchanges = get_exchanges_list()

# Возьмём только первые 3 биржи для теста
gen = volume_generator(exchanges, max_exchanges=3)

for i, message in enumerate(gen, 1):
    print(f"\n--- Сообщение {i} ---")
    print(message[:200], "...")          # первые 200 байт
    data = json.loads(message.decode())
    print("Ключ (биржа):", list(data.keys())[0])
    print("Количество точек объёма:", len(list(data.values())[0]))


--- Сообщение 1 ---
b'{"10KSwap": [[1785758400000, "0.1272261247910159"], [1785759000000, "0.1215739462529598"], [1785759600000, "0.1215739462529598"], [1785760200000, "0.1214599959149405"], [1785760800000, "0.121459995914' ...
Ключ (биржа): 10KSwap
Количество точек объёма: 143

--- Сообщение 2 ---
b'{"9inch": [[1785758400000, "0.5490616940072902"], [1785759000000, "0.5555335148061985"], [1785759600000, "0.5555335148061985"], [1785760200000, "0.5574485600745857"], [1785760800000, "0.55744856007458' ...
Ключ (биржа): 9inch
Количество точек объёма: 143

--- Сообщение 3 ---
b'{"9mm V3 (Pulsechain)": [[1785758400000, "11.2587565922918155"], [1785759000000, "11.2587565922918155"], [1785759600000, "11.1128778196932247"], [1785760200000, "11.1128778196932247"], [1785760800000,' ...
Ключ (биржа): 9mm V3 (Pulsechain)
Количество точек объёма: 144
